# 01 - Entendimento dos Dados

Objetivo: carregar os arquivos brutos do Kaggle, validar disponibilidade, unir as fontes e inspecionar qualidade básica antes de modelar.

Boas práticas usadas aqui:

- Reutilizar o código de produção em `src/`.
- Não criar features de modelo nesta etapa.
- Confirmar distribuição da variável alvo e cobertura temporal.
- Evitar usar acurácia como referência de performance.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config.settings import Settings
from src.data.load_data import RawDataRepository
from src.data.merge_data import FraudDataMerger
from src.features.cleaning import FraudDataCleaner

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

settings = Settings(project_root=PROJECT_ROOT)
settings.raw_data_dir

## 1. Verificar arquivos esperados

Baixe o dataset do Kaggle e coloque os arquivos em `data/raw`.

In [ ]:
expected_files = [
    "transactions_data.csv",
    "cards_data.csv",
    "users_data.csv",
    "mcc_codes.json",
    "train_fraud_labels.json",
]

file_status = pd.DataFrame(
    {
        "file": expected_files,
        "exists": [(settings.raw_data_dir / name).exists() for name in expected_files],
        "path": [str(settings.raw_data_dir / name) for name in expected_files],
    }
)
file_status

## 2. Carregar fontes brutas

In [ ]:
repo = RawDataRepository(settings)
raw = repo.load_all()

shape_summary = []
for name, value in raw.items():
    if isinstance(value, pd.DataFrame):
        shape_summary.append({"source": name, "rows": value.shape[0], "columns": value.shape[1]})
    else:
        shape_summary.append({"source": name, "rows": len(value) if hasattr(value, "__len__") else None, "columns": None})

pd.DataFrame(shape_summary)

In [ ]:
raw["transactions"].head()

## 3. Unir dados e limpar campos básicos

In [ ]:
merged = FraudDataMerger(settings).merge(
    transactions=raw["transactions"],
    cards=raw["cards"],
    users=raw["users"],
    mcc_codes=raw["mcc"],
    labels=raw["labels"],
)
cleaned = FraudDataCleaner(settings).fit_transform(merged)

cleaned.shape, cleaned.head()

## 4. Distribuição da fraude

Fraude costuma ser um problema desbalanceado. Por isso, acurácia não é uma métrica principal adequada.

In [ ]:
target = settings.target_column
target_distribution = (
    cleaned[target]
    .value_counts(dropna=False)
    .rename_axis("is_fraud")
    .reset_index(name="count")
)
target_distribution["rate"] = target_distribution["count"] / target_distribution["count"].sum()
target_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=target_distribution, x="is_fraud", y="count", ax=ax)
ax.set_title("Distribuição da variável alvo")
ax.set_xlabel("Fraude")
ax.set_ylabel("Quantidade")
plt.show()

## 5. Cobertura temporal e taxa de fraude por mês

In [ ]:
from src.data.merge_data import first_existing

time_col = first_existing(cleaned.columns, settings.time_column_candidates)
time_col

In [ ]:
cleaned[time_col].agg(["min", "max"])

In [ ]:
monthly = (
    cleaned.assign(month=cleaned[time_col].dt.to_period("M").dt.to_timestamp())
    .groupby("month")
    .agg(transactions=(target, "size"), fraud_rate=(target, "mean"))
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
sns.lineplot(data=monthly, x="month", y="transactions", ax=axes[0])
sns.lineplot(data=monthly, x="month", y="fraud_rate", ax=axes[1])
axes[0].set_title("Volume mensal de transações")
axes[1].set_title("Taxa mensal de fraude")
axes[1].set_ylabel("fraud_rate")
plt.tight_layout()
plt.show()

## 6. Qualidade dos dados

Inspecione nulos, tipos e cardinalidade. Alta cardinalidade em identificadores crus não deve entrar como feature.

In [ ]:
quality = pd.DataFrame(
    {
        "dtype": cleaned.dtypes.astype(str),
        "missing_rate": cleaned.isna().mean(),
        "nunique": cleaned.nunique(dropna=True),
    }
).sort_values("missing_rate", ascending=False)
quality.head(30)